# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SDlel/ml-assignments/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The playbook starts with the ranked `final_refresh_score`, then shows a suggested action, confidence label, and reason codes. The reason codes are meant to explain why a page deserves review, not to replace a human content decision.

In [1]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "scripts").exists())

WORK_OUTPUTS = ROOT / "work" / "outputs"
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import display_path


def run_script(script_name):
    subprocess.run([sys.executable, str(ROOT / "scripts" / script_name)], cwd=ROOT, check=True)

for script in ["01_prepare_features.py", "02_baseline_score.py", "03_train_model.py", "04_evaluate_and_export.py"]:
    run_script(script)

queue = pd.read_csv(ROOT / "outputs" / "refresh_queue.csv")
summary = json.loads((ROOT / "outputs" / "summary.json").read_text())
queue_export = WORK_OUTPUTS / "refresh_action_playbook.csv"
queue.to_csv(queue_export, index=False)

queue.head(10)[[
    "final_rank", "content_id", "final_refresh_score", "confidence",
    "suggested_action", "final_reason_codes", "impressions_90d", "sessions_90d", "trend_direction"
]]

,final_rank,content_id,final_refresh_score,confidence,suggested_action,final_reason_codes,impressions_90d,sessions_90d,trend_direction
0,1,content_1f080331fa2b,81.928467,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,66,down
1,2,content_6aa43079fb0c,81.728449,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,8064,23,down
2,3,content_d6570c51c9bd,81.639118,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2498,9,down
3,4,content_e04eb9549989,80.804986,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,3393,5,down
4,5,content_72e800a9c214,80.801530,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,13790,27,down
5,6,content_9b6df29f7889,80.752578,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1622,20,down
6,7,content_b69288c5e701,80.632372,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,5811,14,down
7,8,content_ba6f9dfcbca1,80.439403,medium,refresh,declining_with_demand|model_decline_risk|visib...,4366,8,down
8,9,content_b1d593faf9c6,80.204325,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,2655,92,down
9,10,content_bb6ebb5ec8c8,80.146808,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2621,10,down


## 2. Intended use and limits

*Who uses this, for what ? and where it stops being valid.*

A content or SEO reviewer would use this to choose which pages to inspect first during a refresh cycle. It is valid as directional prioritization on the bundled anonymized data, but it should not be used to auto-publish edits, infer client identity, or claim search ranking causality.

In [2]:
use_limits = pd.DataFrame([
    {"area": "intended user", "note": "content or SEO reviewer"},
    {"area": "intended use", "note": "prioritize manual refresh review"},
    {"area": "not valid for", "note": "automatic publishing or causal ranking claims"},
    {"area": "data boundary", "note": "bundled anonymized sample only"},
])
use_limits.to_csv(WORK_OUTPUTS / "playbook_use_limits.csv", index=False)
use_limits

,area,note
0,intended user,content or SEO reviewer
1,intended use,prioritize manual refresh review
2,not valid for,automatic publishing or causal ranking claims
3,data boundary,bundled anonymized sample only


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting, a reviewer should check the live page, current SERP intent, content accuracy, internal priorities, and whether the page already has a scheduled update. The no-go list is automatic rewrites, client-identifying exports, claims that the model predicts Google, and any decision based only on one score.

In [3]:
review_steps = pd.DataFrame([
    {"step": 1, "check": "Open the live page and confirm the content still exists."},
    {"step": 2, "check": "Compare the recommendation with current search intent."},
    {"step": 3, "check": "Check whether low CTR or engagement has a non-content explanation."},
    {"step": 4, "check": "Confirm the page is allowed and useful to refresh now."},
])
no_go = pd.DataFrame({"never_automate": [
    "publishing rewrites",
    "removing pages",
    "client-identifying exports",
    "causal ranking claims",
]})
review_steps.to_csv(WORK_OUTPUTS / "human_review_steps.csv", index=False)
no_go.to_csv(WORK_OUTPUTS / "no_go_list.csv", index=False)
review_steps, no_go

(   step                                              check
 0     1  Open the live page and confirm the content sti...
 1     2  Compare the recommendation with current search...
 2     3  Check whether low CTR or engagement has a non-...
 3     4  Confirm the page is allowed and useful to refr...,
                never_automate
 0         publishing rewrites
 1              removing pages
 2  client-identifying exports
 3       causal ranking claims)

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

I would retrain or re-audit if the label rate shifts a lot, Precision@50 drops on a fresh holdout, the mix of suggested actions changes sharply, or the data contract changes. I would also re-check features if search measurement windows or content-type definitions change.

In [4]:
action_mix = queue["suggested_action"].value_counts(normalize=True).rename("share").reset_index().rename(columns={"index": "suggested_action"})
confidence_mix = queue["confidence"].value_counts(normalize=True).rename("share").reset_index().rename(columns={"index": "confidence"})
monitoring = pd.DataFrame([
    {"trigger": "precision_at_50 drop", "threshold": "material drop on a new holdout"},
    {"trigger": "label-rate shift", "threshold": "large movement from the measured sample rate"},
    {"trigger": "action mix shift", "threshold": "one action category suddenly dominates"},
    {"trigger": "data contract change", "threshold": "new or redefined fields in the feature set"},
])
monitoring.to_csv(WORK_OUTPUTS / "monitoring_triggers.csv", index=False)
action_mix.round(3), confidence_mix.round(3), monitoring

(                suggested_action  share
 0                        monitor  0.436
 1                        refresh  0.274
 2         refresh_and_review_ctr  0.222
 3  refresh_and_review_engagement  0.066
 4             expand_and_refresh  0.003,
   confidence  share
 0        low  0.500
 1     medium  0.381
 2       high  0.119,
                 trigger                                     threshold
 0  precision_at_50 drop                material drop on a new holdout
 1      label-rate shift  large movement from the measured sample rate
 2      action mix shift        one action category suddenly dominates
 3  data contract change    new or redefined fields in the feature set)

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ ? your paper builds on these receipts.*

The notebook exports a final action queue plus small summary tables for the capstone. The charts are generated by the pipeline and copied into `work/outputs/charts/` for the paper.

In [5]:
charts_dir = WORK_OUTPUTS / "charts"
charts_dir.mkdir(parents=True, exist_ok=True)
for chart_path in (ROOT / "outputs" / "charts").glob("*.svg"):
    shutil.copy2(chart_path, charts_dir / chart_path.name)

paper_receipts = pd.DataFrame([
    {"artifact": "refresh_action_playbook.csv", "path": display_path(queue_export)},
    {"artifact": "model_report.md", "path": "outputs/model_report.md"},
    {"artifact": "model_results.json", "path": "outputs/model_results.json"},
    {"artifact": "summary.json", "path": "outputs/summary.json"},
    {"artifact": "charts", "path": display_path(charts_dir)},
])
paper_receipts.to_csv(WORK_OUTPUTS / "paper_receipts.csv", index=False)
paper_receipts

,artifact,path
0,refresh_action_playbook.csv,work\outputs\refresh_action_playbook.csv
1,model_report.md,outputs/model_report.md
2,model_results.json,outputs/model_results.json
3,summary.json,outputs/summary.json
4,charts,work\outputs\charts


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled ? markdown thinking AND code where code is requested.
- [x] Every number comes from a query, dataframe, or saved output.
- [x] I used careful language: observed / measured / directional / decision-support.
- [x] I did not include private client names, domains, URLs, titles, or keywords.